# Step 4. Transfer

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/younesStrittmatter/sweetpea-starter/blob/main/tutorial/notebooks/4_transfer.ipynb)

The tutorial step without an assistant. Run the cells from top to bottom; the tables below are from one run, yours will differ where the design leaves things open.

In [ ]:
%pip install -q sweetpea pandas

**So far.** Step 2 made the trial before a factor. Step 3 made blocks that are mostly congruent or mostly incongruent, generated one by one. This step uses both ideas one level up: the mini-block is the unit, and the whole session is one design.

**The question.** Does adapting to frequent conflict make it harder to change the task?

**Prediction if it is one control setting.** Where conflict is frequent the current task is shielded more, so a change of task costs more: the switch cost is larger in the mostly incongruent context.

**Why word reading.** The switch cost is taken on word-reading mini-blocks. Word reading has almost no congruency effect, so the proportion itself hardly touches the trials the cost is measured on.

**The analysis.** Switch cost = response time in word-reading mini-blocks that follow colour naming (switch) minus response time in word-reading mini-blocks that follow word reading (repeat). Compare it between the two contexts.

### As before, plus the task and the context

Both are regular factors. Neither changes from trial to trial: each is held for a whole mini-block.

In [ ]:
import pandas as pd
from sweetpea import (Factor, DerivedLevel, WithinTrial, Transition, Exclude, MinimumTrials,
                      CrossBlock, synthesize_trials, CMSGen)

colours = ["red", "green", "blue"]

# regular factors: set freely on every trial
colour = Factor("colour", colours)      # the ink
word = Factor("word", colours)          # what the word says
from sweetpea import Nest, IterateGen

context = Factor("context", ["mostly congruent", "mostly incongruent"])  # new: held for a whole mini-block
task = Factor("task", ["name the colour", "read the word"])              # new: held for a whole mini-block

### New: what happens at the border between two mini-blocks?

Two transition factors, as in step 2. Here they will look back one mini-block, not one trial: does the task change, and which context came before?

In [ ]:
def task_repeats(task):
    return task[0] == task[-1]


def task_switches(task):
    return task[0] != task[-1]


task_transition = Factor("task_transition", [
    DerivedLevel("repeat", Transition(task_repeats, [task])),
    DerivedLevel("switch", Transition(task_switches, [task])),
])


def after_mostly_congruent(context):
    return context[-1] == "mostly congruent"


def after_mostly_incongruent(context):
    return context[-1] == "mostly incongruent"


context_before = Factor("context_before", [
    DerivedLevel("mostly congruent", Transition(after_mostly_congruent, [context])),
    DerivedLevel("mostly incongruent", Transition(after_mostly_incongruent, [context])),
])

### New: what the context makes frequent

A weight makes a level more frequent: the third argument of a level, and 3 to 1 is three quarters. But a level has one weight, and here the proportion depends on the context. SweetPea has no weight that depends on another factor. So the weight sits on a helper factor that says whether a trial is the kind its context makes frequent: congruent in a mostly congruent mini-block, incongruent in a mostly incongruent one.

In [ ]:
def frequent_here(colour, word, context):
    congruent = colour == word
    return congruent == (context == "mostly congruent")


def rare_here(colour, word, context):
    return not frequent_here(colour, word, context)


frequency = Factor("frequency", [
    DerivedLevel("frequent", WithinTrial(frequent_here, [colour, word, context]), 3),
    DerivedLevel("rare", WithinTrial(rare_here, [colour, word, context]), 1),
])

### New: a block inside a block

Two blocks, each with its own crossing:

- **within a mini-block**: the frequency crossed with the ink colour, 12 trials;
- **between mini-blocks**: the context, the context before, the task and the task transition. That is every kind of border, 16 mini-blocks, plus a first one that has no mini-block before it.

`Nest` puts the first inside the second: every combination of the outer block is held for one whole mini-block.

In [ ]:
mini_block = CrossBlock(design=[colour, word, frequency, context],
                        crossing=[frequency, colour],
                        constraints=[])
between_mini_blocks = CrossBlock(design=[context, context_before, task, task_transition],
                                 crossing=[context, context_before, task, task_transition],
                                 constraints=[])
block = Nest(outer_block=between_mini_blocks, inner_block=mini_block, alignment="post preamble")
session = pd.DataFrame(synthesize_trials(block, 1, IterateGen)[0])       # 204 trials: too long for CMSGen

# columns for counting
session["congruency"] = ["congruent" if colour == word else "incongruent"
                         for colour, word in zip(session["colour"], session["word"])]
session.insert(0, "block", session.index // 12 + 1)
overview = session.groupby("block")[["context_before", "context", "task", "task_transition"]].first()
overview["congruent trials"] = session.groupby("block")["congruency"].apply(lambda c: (c == "congruent").sum())
display(overview)

,context_before,context,task,task_transition,congruent trials
block,,,,,
1,,mostly incongruent,name the colour,,5
2,mostly incongruent,mostly incongruent,read the word,switch,3
3,mostly incongruent,mostly incongruent,name the colour,switch,3
4,mostly incongruent,mostly incongruent,name the colour,repeat,3
5,mostly incongruent,mostly congruent,read the word,switch,9
6,mostly congruent,mostly incongruent,read the word,repeat,3
7,mostly incongruent,mostly incongruent,read the word,repeat,3
8,mostly incongruent,mostly congruent,name the colour,switch,9
9,mostly congruent,mostly congruent,name the colour,repeat,9


One row per mini-block. The first has nothing before it; it is the start, and its trials are not crossed. After it, every mini-block has the proportion of its context: 9 or 3 of 12.

### Between mini-blocks: is every kind of border there?

In [ ]:
counted = session[session["task_transition"] != ""]          # without the first mini-block
mini_blocks = counted.groupby("block")[["context_before", "context", "task", "task_transition"]].first()
display(pd.crosstab([mini_blocks["context_before"], mini_blocks["context"]], [mini_blocks["task"], mini_blocks["task_transition"]]))

task                                  name the colour  ...       
task_transition                                repeat  ... switch
context_before     context                             ...       
mostly congruent   mostly congruent                 1  ...      1
                   mostly incongruent               1  ...      1
mostly incongruent mostly congruent                 1  ...      1
                   mostly incongruent               1  ...      1

[4 rows x 4 columns]

### Within a mini-block: is the ink colour even?

In [ ]:
display(pd.crosstab([counted["block"], counted["context"]], [counted["congruency"], counted["colour"]]))

congruency               congruent           incongruent          
colour                        blue green red        blue green red
block context                                                     
2     mostly incongruent         1     1   1           3     3   3
3     mostly incongruent         1     1   1           3     3   3
4     mostly incongruent         1     1   1           3     3   3
5     mostly congruent           3     3   3           1     1   1
6     mostly incongruent         1     1   1           3     3   3
7     mostly incongruent         1     1   1           3     3   3
8     mostly congruent           3     3   3           1     1   1
9     mostly congruent           3     3   3           1     1   1
10    mostly incongruent         1     1   1           3     3   3
11    mostly congruent           3     3   3           1     1   1
12    mostly congruent           3     3   3           1     1   1
13    mostly congruent           3     3   3           1     1   1
14    mostly congruent           3     3   3           1     1   1
15    mostly incongruent         1     1   1           3     3   3
16    mostly congruent           3     3   3           1     1   1
17    mostly incongruent         1     1   1           3     3   3

### What is not in the design?

The two trials at the border themselves: the last trial of one mini-block and the first trial of the next. Count them for the mini-blocks where the task switches.

In [ ]:
first = session.groupby("block")["congruency"].first().rename("first trial after the border")
last_before = session.groupby("block")["congruency"].last().shift().rename("last trial before the border")
switches = mini_blocks[mini_blocks["task_transition"] == "switch"].index
display(pd.crosstab(last_before[switches], first[switches]))

first trial after the border,congruent,incongruent
last trial before the border,,
congruent,3,3
incongruent,2,0


**Check.** Both crossings are exact, and every kind of border occurs once. What is thin? Each kind of border is one mini-block of 12 trials; more need several sequences. The two trials at the border are left to chance: is that acceptable, or should the first trial of a mini-block be pinned?

**The sampler.** `CMSGen` did not finish this design in five minutes. `IterateGen` takes a few seconds, but it does not draw at random among all possible sequences: it returns the solutions in the order the solver finds them. Look at the order of the trials before you use it.

**Try.** Take `context_before` out of the outer block. How many mini-blocks are left, and does `CMSGen` work again?

**The idea.** Each question added a factor, and each factor multiplies the design. Nesting keeps both levels counterbalanced at once. What it leaves open is still a choice, and which choice is acceptable depends on what you compare.